# Member 4 — Gradient Boosting

Owner: IT24104068

Purpose: Train and tune a Gradient Boosting classifier.

## 1. Load Data and Saved Training Split

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import make_scorer, precision_score
from sklearn.base import clone

data_path = Path("../../data/processed/hotel_bookings_validated_phase3.csv")
split_path = Path("../../results/splits/group_holdout_indices.npz")
df = pd.read_csv(data_path)
splits = np.load(split_path)

excluded = [
    "is_canceled", "assigned_room_type", "booking_changes",
    "deposit_type", "agent", "company", "adr",
    "required_car_parking_spaces", "total_of_special_requests",
    "reservation_status", "reservation_status_date"
]
X = df.drop(columns=excluded)
y = df["is_canceled"]
X_train = X.iloc[splits["train_idx"]].copy()
y_train = y.iloc[splits["train_idx"]].copy()

print("Training shape:", X_train.shape)
print("Held-out test rows:", len(splits["test_idx"]))
print("Selected features:", X_train.shape[1])
print("Training cancellation rate:", f"{y_train.mean():.2%}")


## 2. Original Gradient Boosting Model

In [ ]:
repo_root = data_path.resolve().parents[2]
sys.path.insert(0, str(repo_root))
from src.preprocessing.pipeline import build_preprocessor

gb_model = Pipeline([
    ("preprocessing", build_preprocessor(X_train)),
    ("classifier", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        min_samples_leaf=20,
        subsample=0.8,
        max_features="sqrt",
        random_state=42
    ))
])

groups_train = pd.util.hash_pandas_object(X_train, index=False)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}
print("Original Gradient Boosting and group-aware evaluation configured.")


In [ ]:
gb_scores = cross_validate(
    gb_model, X_train, y_train,
    groups=groups_train, cv=cv, scoring=scoring,
    n_jobs=1, return_train_score=True
)

print("Gradient Boosting - 5-Fold Cross-Validation\n")
for metric in scoring:
    print(f"{metric}: {gb_scores[f'test_{metric}'].mean():.4f}")


Gradient Boosting - 5-Fold Cross-Validation

accuracy: 0.7844
precision: 0.8213
recall: 0.5331
f1: 0.6463
roc_auc: 0.8655
pr_auc: 0.8177


In [ ]:
train_f1 = gb_scores["train_f1"].mean()
validation_f1 = gb_scores["test_f1"].mean()
print("Gradient Boosting - Overfitting Check")
print(f"Training F1-score: {train_f1:.4f}")
print(f"Validation F1-score: {validation_f1:.4f}")
print(f"F1-score gap: {train_f1 - validation_f1:.4f}")

output_dir = Path("../../results/metrics")
output_dir.mkdir(parents=True, exist_ok=True)
original_results = pd.DataFrame({
    metric: gb_scores[f"test_{metric}"] for metric in scoring
})
original_results.index = [f"Fold {i}" for i in range(1, 6)]
original_results.loc["Mean"] = original_results.mean()
original_results.to_csv(output_dir / "gradient_boosting_cv.csv", index_label="Fold")
print("Original evaluation results saved.")


Gradient Boosting - Overfitting Check
Training F1-score: 0.6508
Validation F1-score: 0.6463
F1-score gap: 0.0045


## Gradient Boosting Evaluation Summary

Gradient Boosting achieved 78.44% accuracy, 82.13% precision, 53.31% recall, and an F1-score of 0.6463.

The model recorded a ROC-AUC of 0.8655 and PR-AUC of 0.8177, indicating good cancellation-risk discrimination.

The training F1-score (0.6508) and validation F1-score (0.6463) showed a small gap of 0.0045, suggesting limited evidence of overfitting.

Gradient Boosting will be included in the final model comparison. Model selection will consider predictive performance, cancellation recall, and practical business requirements.

## 3. Hyperparameter Tuning

In [ ]:
param_distributions = {
    "classifier__n_estimators": [80, 120],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_depth": [2, 3],
    "classifier__min_samples_leaf": [10, 20]
}
tuning_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
gb_search = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=param_distributions,
    n_iter=4,
    scoring="average_precision",
    cv=tuning_cv,
    random_state=42,
    n_jobs=1
)
print("Configurations:", gb_search.n_iter)
print("Tuning folds:", tuning_cv.n_splits)


In [ ]:
gb_search.fit(X_train, y_train, groups=groups_train)

print("Gradient Boosting - Hyperparameter Tuning Results\n")
print("Best hyperparameters:")
for param, value in gb_search.best_params_.items():
    print(f"{param}: {value}")
print(f"\nBest cross-validation PR-AUC: {gb_search.best_score_:.4f}")

gb_tuning_results = pd.DataFrame(gb_search.cv_results_)
columns = [
    "param_classifier__n_estimators",
    "param_classifier__learning_rate",
    "param_classifier__max_depth",
    "param_classifier__min_samples_leaf",
    "mean_test_score", "std_test_score"
]
display(gb_tuning_results[columns].sort_values("mean_test_score", ascending=False).round(4))


Gradient Boosting - Hyperparameter Tuning Results

Best hyperparameters:
classifier__n_estimators: 80
classifier__min_samples_leaf: 20
classifier__max_depth: 3
classifier__learning_rate: 0.1

Best cross-validation PR-AUC: 0.8100


,param_classifier__n_estimators,param_classifier__learning_rate,param_classifier__max_depth,param_classifier__min_samples_leaf,mean_test_score,std_test_score
3,80,0.10,3,20,0.8100,0.0176
2,120,0.05,3,10,0.8090,0.0171
1,120,0.05,2,10,0.7950,0.0224
0,80,0.05,2,10,0.7863,0.0256


## 4. Tuned Model Evaluation and Comparison

In [ ]:
tuned_gb_model = clone(gb_model).set_params(**gb_search.best_params_)
tuned_gb_scores = cross_validate(
    tuned_gb_model, X_train, y_train,
    groups=groups_train, cv=cv, scoring=scoring,
    n_jobs=1, return_train_score=True
)
print("Tuned Gradient Boosting cross-validation completed successfully.")


Tuned Gradient Boosting cross-validation completed successfully.


In [ ]:
comparison = pd.DataFrame({
    "Original": {metric: gb_scores[f"test_{metric}"].mean() for metric in scoring},
    "Tuned": {metric: tuned_gb_scores[f"test_{metric}"].mean() for metric in scoring}
}).T
display(comparison.round(4))

for name, scores in [("Original", gb_scores), ("Tuned", tuned_gb_scores)]:
    train_f1 = scores["train_f1"].mean()
    val_f1 = scores["test_f1"].mean()
    print(f"{name} Gradient Boosting")
    print(f"Training F1: {train_f1:.4f}")
    print(f"Validation F1: {val_f1:.4f}")
    print(f"F1 gap: {train_f1 - val_f1:.4f}\n")


,accuracy,precision,recall,f1,roc_auc,pr_auc
Original,0.7844,0.8213,0.5331,0.6463,0.8655,0.8177
Tuned,0.7790,0.8240,0.5116,0.6310,0.8616,0.8139


Original Gradient Boosting
Training F1: 0.6508
Validation F1: 0.6463
F1 gap: 0.0045

Tuned Gradient Boosting
Training F1: 0.6379
Validation F1: 0.6310
F1 gap: 0.0069



In [ ]:
gb_tuning_results.sort_values("rank_test_score").to_csv(
    output_dir / "gradient_boosting_tuning.csv", index=False
)
tuned_results = pd.DataFrame({
    metric: tuned_gb_scores[f"test_{metric}"] for metric in scoring
})
tuned_results.index = [f"Fold {i}" for i in range(1, 6)]
tuned_results.loc["Mean"] = tuned_results.mean()
tuned_results.round(4).to_csv(
    output_dir / "gradient_boosting_tuned_cv.csv", index_label="Fold"
)
print("Tuning and tuned-model evaluation results saved.")



## Hyperparameter Tuning Summary

Hyperparameter tuning was performed using RandomizedSearchCV with three-fold StratifiedGroupKFold cross-validation, optimizing PR-AUC on the training data.

The best configuration among the four tested combinations was:
- `n_estimators = 80`
- `learning_rate = 0.1`
- `max_depth = 3`
- `min_samples_leaf = 20`

Five-fold cross-validation showed that the original model performed better than the tuned configuration:

| Metric | Original | Tuned |
|---|---:|---:|
| Precision | 0.8213 | 0.8240 |
| Recall | 0.5331 | 0.5116 |
| F1-score | 0.6463 | 0.6310 |
| PR-AUC | 0.8177 | 0.8139 |

The original model also showed a smaller training–validation F1 gap (0.0045 versus 0.0069), indicating limited overfitting.

**Decision:** Retain the original Gradient Boosting model with 100 estimators. Hyperparameter tuning did not improve validation performance. The model remains a candidate for final comparison with the other classification models.
